# Immigration Enforcement in Texas Network and Routing Analysis

In [1]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import contextily as cx
import os
import pandas as pd
from multiprocessing import Pool, Process, Queue, Lock, Value, Array, Manager

In [2]:
os.getcwd()

'/home/jovyan/work/GGIS 407/Final Project'

In [3]:
os.listdir("/home/jovyan/work/GGIS 407/Final Project/data")

['facilities.csv',
 'G_drive_harlingen.graphml',
 'geojson-counties-fips.json',
 'EOIR scraper.ipynb',
 'pgdata',
 '.ipynb_checkpoints',
 'ice-aor-county-shp.shx',
 'tx_287g_counties_gdf.geojson',
 'immigration_court_administrative_control.csv',
 'ice-aor-county-shp.prj',
 'participating_pendingAgencies02122026pm.csv',
 'courts_gdf.geojson',
 'download_contracts.csv',
 'processed',
 'ice-aor-county-shp.shp',
 'texas_immigration_judges.csv',
 'ice-aor-county-shp.dbf',
 'EOIR_complete_gdf.geojson',
 'EFF_towers_01_26.csv']

## Initialize PostGIS Database and Import Datasets

In [4]:
os.environ["PGDATA"] = "/home/jovyan/work/GGIS 407/Final Project/data/pgdata"
!pg_ctl initdb
!pg_ctl start
%reload_ext sql
!createdb tx_immigration_enforcement
%sql postgresql://localhost:5432/tx_immigration_enforcement

The files belonging to this database system will be owned by user "jovyan".
This user must also own the server process.

The database cluster will be initialized with locale "en_US.UTF-8".
The default database encoding has accordingly been set to "UTF8".
The default text search configuration will be set to "english".

Data page checksums are disabled.

initdb: error: directory "/home/jovyan/work/GGIS 407/Final Project/data/pgdata" exists but is not empty
If you want to create a new database system, either remove or empty
the directory "/home/jovyan/work/GGIS 407/Final Project/data/pgdata" or run initdb
with an argument other than "/home/jovyan/work/GGIS 407/Final Project/data/pgdata".
pg_ctl: database system initialization failed
pg_ctl: another server might be running; trying to start server anyway
waiting for server to start....2026-03-09 18:06:57.455 UTC [11727] FATAL:  lock file "postmaster.pid" already exists
2026-03-09 18:06:57.455 UTC [11727] HINT:  Is another postmaster (PID 19

In [5]:
%%sql
drop extension if exists postgis cascade;
create extension postgis;

 * postgresql://localhost:5432/tx_immigration_enforcement
Done.
Done.


[]

In [6]:
!shp2pgsql -s 4269 -d -I "/home/jovyan/work/GGIS 407/Final Project/data/ice-aor-county-shp.shp" public.ice_aor | psql -q -d tx_immigration_enforcement

Field aland is an FTDouble with width 24 and precision 15
Field awater is an FTDouble with width 24 and precision 15
Shapefile type: Polygon
Postgis type: MULTIPOLYGON[2]
ERROR:  column not found in geometry_columns table
CONTEXT:  PL/pgSQL function dropgeometrycolumn(character varying,character varying,character varying,character varying) line 33 at RAISE
SQL statement "SELECT public.DropGeometryColumn('',$1,$2,$3)"
PL/pgSQL function dropgeometrycolumn(character varying,character varying,character varying) line 5 at SQL statement
                    addgeometrycolumn                    
---------------------------------------------------------
 public.ice_aor.geom SRID:4269 TYPE:MULTIPOLYGON DIMS:2 
(1 row)



In [7]:
# Import GeoJSON files into PostgreSQL database
!ogr2ogr -f "PostgreSQL" PG:"dbname=tx_immigration_enforcement host=localhost port=5432" \
    "/home/jovyan/work/GGIS 407/Final Project/data/processed/courts_gdf.geojson" \
    -nln public.courts \
    -t_srs EPSG:4269 \
    -overwrite

!ogr2ogr -f "PostgreSQL" PG:"dbname=tx_immigration_enforcement host=localhost port=5432" \
    "/home/jovyan/work/GGIS 407/Final Project/data/processed/EOIR_complete_gdf.geojson" \
    -nln public.detention_centers \
    -t_srs EPSG:4269 \
    -overwrite

!ogr2ogr -f "PostgreSQL" PG:"dbname=tx_immigration_enforcement host=localhost port=5432" \
    "/home/jovyan/work/GGIS 407/Final Project/data/processed/tx_287g_counties_gdf.geojson" \
    -nln public.counties_287g \
    -t_srs EPSG:4269 \
    -overwrite

In [8]:
%sql \d

 * postgresql://localhost:5432/tx_immigration_enforcement
14 rows affected.


Schema,Name,Type,Owner
public,aor_to_counties,table,jovyan
public,counties_287g,table,jovyan
public,counties_287g_ogc_fid_seq,sequence,jovyan
public,courts,table,jovyan
public,courts_ogc_fid_seq,sequence,jovyan
public,detention_centers,table,jovyan
public,detention_centers_ogc_fid_seq,sequence,jovyan
public,detention_facilities,table,jovyan
public,geography_columns,view,jovyan
public,geometry_columns,view,jovyan


In [9]:
%%sql
SELECT column_name, data_type 
FROM information_schema.columns 
WHERE table_name = 'detention_centers';

 * postgresql://localhost:5432/tx_immigration_enforcement
9 rows affected.


column_name,data_type
ogc_fid,integer
facility_name,character varying
facility_type,character varying
administrative_control_court,character varying
court_address,character varying
judge_name,character varying
latitude,double precision
longitude,double precision
wkb_geometry,USER-DEFINED


In [10]:
%%sql
select *
from geometry_columns

 * postgresql://localhost:5432/tx_immigration_enforcement
4 rows affected.


f_table_catalog,f_table_schema,f_table_name,f_geometry_column,coord_dimension,srid,type
tx_immigration_enforcement,public,ice_aor,geom,2,4269,MULTIPOLYGON
tx_immigration_enforcement,public,courts,wkb_geometry,2,4269,POINT
tx_immigration_enforcement,public,detention_centers,wkb_geometry,2,4269,POINT
tx_immigration_enforcement,public,counties_287g,wkb_geometry,2,4269,GEOMETRY


## Query Operations

In [11]:
%%sql
drop table if exists aor_to_counties;
create table aor_to_counties as
select 
    name AS county, 
    aor_nam, 
    offc_nm, 
    geom, 
    aland, 
    awater
from ice_aor
where state_n = 'Texas'
order by aor_nam

 * postgresql://localhost:5432/tx_immigration_enforcement
Done.
254 rows affected.


[]

### Create table: `detention_facilities`

In [12]:
%%sql 
drop table if exists detention_facilities;
create table detention_facilities as
select 
    facility_name, 
    facility_type, 
    administrative_control_court, 
    court_address, 
    ST_AsText(wkb_geometry) as wkb_geometry
FROM detention_centers
WHERE facility_name IS NOT NULL

 * postgresql://localhost:5432/tx_immigration_enforcement
Done.
55 rows affected.


[]

In [13]:
%%sql
with tbl as (
    select 
        administrative_control_court, 
        count(facility_name) as total_facility_count
    from detention_facilities 
    group by administrative_control_court
)

select 
    d.administrative_control_court, 
    d.facility_type, 
    count(d.facility_name) as facility_type_count, 
    tbl.total_facility_count
from detention_facilities as d
left join tbl on d.administrative_control_court = tbl.administrative_control_court 
group by d.administrative_control_court, d.facility_type, tbl.total_facility_count
order by d.administrative_control_court

 * postgresql://localhost:5432/tx_immigration_enforcement
36 rows affected.


administrative_control_court,facility_type,facility_type_count,total_facility_count
Conroe Immigration Court,Detention Facility,5,7
Conroe Immigration Court,Processing Center,1,7
Conroe Immigration Court,Service Processing Center,1,7
Dallas Immigration Court,DHS Office,1,3
Dallas Immigration Court,Juvenile Facility,2,3
El Paso Immigration Court,Detention Facility,1,4
El Paso Immigration Court,DHS Office,1,4
El Paso Immigration Court,Port of Entry,1,4
El Paso Immigration Court,Processing Center,1,4
El Paso SPC Immigration Court,Detention Facility,1,3


### Create table: `judge_details`

In [14]:
%%sql 
select 
    administrative_control_court, 
    court_address, 
    total_judges,
    ST_AsText(wkb_geometry), 
    wkb_geometry
FROM courts 

 * postgresql://localhost:5432/tx_immigration_enforcement
12 rows affected.


administrative_control_court,court_address,total_judges,st_astext,wkb_geometry
Port Isabel Immigration Court,"27991 Buena Vista Blvd., Los Fresnos, TX 78566",4,POINT(-97.358682873261 26.158641827366),0101000020AD100000A59202A9F45658C08D5034C09C283A40
El Paso SPC Immigration Court,"8915 Montana Ave., Suite 100, El Paso, TX 79925",5,POINT(-106.369208351881 31.793762008564),0101000020AD1000004B2F111CA1975AC0F396ABFC33CB3F40
Harlingen Immigration Court,"2009 West Jefferson Ave., Suite 300, Harlingen, TX 78550",5,POINT(-97.718684038948 26.194727004596),0101000020AD10000016DC56EBFE6D58C04B6304A1D9313A40
Laredo Immigration Court,"1406 Jacaman Rd Suite B, Laredo, Tx 78041",6,POINT(-99.471972072251 27.562795340811),0101000020AD10000061BC59CA34DE58C0DA1FFF5A13903B40
Pearsall Immigration Court,"566 Veterans Drive, Pearsall, TX 78061",7,POINT(-99.120291973986 28.896570461288),0101000020AD1000006F8F1BDDB2C758C0A6CA49A485E53C40
El Paso Immigration Court,"700 East San Antonio Avenue Suite 750, El Paso, TX 79901",8,POINT(-106.482273268738 31.7592035475),0101000020AD100000BE3DB390DD9E5AC00A85E7295BC23F40
Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",10,POINT(-95.442699014881 30.337100979098),0101000020AD100000A9B83F2E55DC57C0EDACF03F4C563E40
Dallas Immigration Court,"1100 Commerce St. Room 1060, Dallas, TX 75242",11,POINT(-96.802110689134 32.778890064349),0101000020AD100000906612C8553358C076C76CABB2634040
Houston - Jefferson Street Immigration Court,"500 Jefferson Street, Suite 300, Houston TX 77002",13,POINT(-95.373800357512 29.75292813487),0101000020AD100000D1AF5558ECD757C04381F3E5BFC03D40
San Antonio Immigration Court,"800 Dolorosa Street Suite 300, San Antonio, TX 78207",13,POINT(-98.498824480222 29.423911194049),0101000020AD100000DA3F83BDEC9F58C049D9AA71856C3D40


In [15]:
%%sql 
drop table if exists judge_details;
create table judge_details as
select 
    judge_name, 
    administrative_control_court, 
    court_address, 
    ST_AsText(wkb_geometry) as geom_astext, 
    wkb_geometry
FROM detention_centers
WHERE judge_name IS NOT NULL

 * postgresql://localhost:5432/tx_immigration_enforcement
Done.
131 rows affected.


[]

In [16]:
%%sql
select 
    administrative_control_court, 
    count(judge_name) as judge_count, 
    geom_astext
from judge_details
group by administrative_control_court, geom_astext
order by judge_count desc

 * postgresql://localhost:5432/tx_immigration_enforcement
13 rows affected.


administrative_control_court,judge_count,geom_astext
Fort Worth IAC Immigration Court,18,None
Houston - Greenspoint Park Immigration Court,17,POINT(-95.402484068369 29.945864458543)
Houston - S. Gessner Road Immigration Court,14,POINT(-95.528444794718 29.686902919638)
San Antonio Immigration Court,13,POINT(-98.498824480222 29.423911194049)
Houston - Jefferson Street Immigration Court,13,POINT(-95.373800357512 29.75292813487)
Dallas Immigration Court,11,POINT(-96.802110689134 32.778890064349)
Conroe Immigration Court,10,POINT(-95.442699014881 30.337100979098)
El Paso Immigration Court,8,POINT(-106.482273268738 31.7592035475)
Pearsall Immigration Court,7,POINT(-99.120291973986 28.896570461288)
Laredo Immigration Court,6,POINT(-99.471972072251 27.562795340811)


### Create table: `county_287g`

In [37]:
%%sql 
select 
    county,
    "law enforcement agency", 
    signed, 
    "jail enforcement model", 
    "task force model", 	
    "warrant service officer",
    ST_AsText(wkb_geometry)
from counties_287g

 * postgresql://localhost:5432/tx_immigration_enforcement
178 rows affected.


county,law enforcement agency,signed,jail enforcement model,task force model,warrant service officer,st_astext
Anderson,Anderson County Sheriff's Office,2025-06-18 00:00:00+00:00,0,0,1,"POLYGON((-95.428512 32.084475,-95.446747 31.843116,-95.412908 31.835157,-95.258859 31.609959,-95.273203 31.592886,-95.651764 31.541791,-95.739279 31.504056,-95.710112 31.615587,-95.7873 31.618385,-95.794081 31.66031,-95.861262 31.687451,-95.875937 31.755503,-95.980568 31.784561,-95.994127 31.866258,-96.027788 31.878242,-96.022955 31.957581,-96.062172 31.95634,-96.052786 32.005895,-95.428512 32.084475))"
Angelina,Angelina County Sheriff's Office,2025-06-18 00:00:00+00:00,0,0,1,"POLYGON((-94.326616 31.224754,-94.129632 31.09928,-94.45782 31.033326,-94.561943 31.058952,-94.842947 31.146578,-94.909502 31.337059,-94.95811 31.38693,-94.959415 31.388884,-94.966254 31.391205,-94.964521 31.395558,-94.967634 31.397412,-94.969369 31.396948,-94.973581 31.399759,-94.97778 31.399381,-94.976068 31.402,-94.976291 31.40525,-94.979364 31.405975,-94.976033 31.407744,-94.983053 31.411593,-94.984753 31.41385,-94.988061 31.414417,-94.990043 31.413356,-94.993832 31.41422,-94.994108 31.417835,-94.997132 31.41678,-94.998247 31.42004,-95.001258 31.417949,-95.005566 31.421349,-95.003345 31.42571,-95.000212 31.428196,-94.865857 31.526916,-94.728679 31.457226,-94.544888 31.431715,-94.530634 31.398654,-94.495874 31.405728,-94.326616 31.224754))"
Armstrong,Armstrong County Sheriff's Office,2026-01-26 00:00:00+00:00,0,1,0,"POLYGON((-101.471562 34.747462,-101.629257 34.747649,-101.629396 34.750056,-101.622941 35.183117,-101.086281 35.18214,-101.090749 34.748246,-101.471562 34.747462))"
Atascosa,Atascosa County Sheriff's Office,2025-05-08 00:00:00+00:00,0,1,0,"POLYGON((-98.800841 28.647487,-98.8049 29.090434,-98.804763 29.250693,-98.407336 29.114435,-98.190991 28.882333,-98.098315 28.786949,-98.335031 28.612658,-98.335047 28.648275,-98.800848 28.647306,-98.800841 28.647487))"
Austin,Austin County Sheriff's Office,2025-04-16 00:00:00+00:00,0,0,1,"POLYGON((-96.62198 30.044283,-96.292849 30.09615,-96.146052 30.070224,-96.084541 30.005137,-96.122754 29.968545,-96.121313 29.841963,-96.049234 29.803187,-96.032711 29.727944,-96.02485 29.602877,-96.088912 29.601658,-96.175422 29.633806,-96.259226 29.668912,-96.344476 29.830147,-96.413283 29.824985,-96.569844 29.961516,-96.62198 30.044283))"
Bailey,Bailey County Sheriff's Office,2025-07-31 00:00:00+00:00,0,0,1,"POLYGON((-103.043979 34.312749,-102.61515 34.312891,-102.615447 33.825121,-103.047346 33.824675,-103.046907 33.8503,-103.045644 33.901537,-103.045698 33.906299,-103.044893 33.945617,-103.04395 33.974629,-103.043617 34.003633,-103.043531 34.018014,-103.043555 34.032714,-103.043746 34.037294,-103.043771 34.041538,-103.043721 34.04232,-103.043767 34.043545,-103.043744 34.049986,-103.043686 34.063078,-103.043516 34.079382,-103.043569 34.087947,-103.043644 34.256903,-103.043719 34.289441,-103.043936 34.302585,-103.043979 34.312749))"
Bell,Holland Police Department,2025-11-04 00:00:00+00:00,0,1,0,"POLYGON((-97.070188 30.98622,-97.259082 30.889596,-97.315507 30.752371,-97.625288 30.87043,-97.828512 30.906188,-97.840365 30.929318,-97.911684 31.034919,-97.913847 31.065882,-97.9071 31.069374,-97.418606 31.320202,-97.343426 31.244215,-97.278113 31.279799,-97.070188 30.98622))"
Bexar,Bexar County Sheriff's Office,2025-10-17 00:00:00+00:00,0,0,1,"POLYGON((-98.28702 29.257626,-98.407336 29.114435,-98.804763 29.250693,-98.806552 29.690709,-98.778782 29.720167,-98.646124 29.745181,-98.550489 29.760713,-98.443852 29.71965,-98.338618 29.721786,-98.378068 29.662613,-98.310928 29.594473,-98.298524 29.561141,-98.272924 29.550878,-98.126767 29.482248,-98.134171 29.441751,-98.137187 29.440428,-98.28702 29.257626))"
Bexar,Bexar County Constable Precinct 3,2025-12-12 00:00:00+00:00,0,1,0,"POLYGON((-98.28702 29.257626,-98.407336 29.114435,-98.804763 29.250693,-98.806552 29.690709,-98.778782 29.720167,-98.646124 29.745181,-98.550489 29.760713,-98.443852 29

In [43]:
%sql select * from detention_facilities where  administrative_control_court = 'Harlingen Immigration Court' limit 10

 * postgresql://localhost:5432/tx_immigration_enforcement
4 rows affected.


facility_name,facility_type,administrative_control_court,court_address,wkb_geometry
"Harlingen, TX - DHS District Office",DHS Office,Harlingen Immigration Court,"2009 West Jefferson Ave., Suite 300, Harlingen, TX 78550",POINT(-97.718684038948 26.194727004596)
Urban Strategies San Benito,Facility,Harlingen Immigration Court,"2009 West Jefferson Ave., Suite 300, Harlingen, TX 78550",POINT(-97.718684038948 26.194727004596)
Brownsville International Gateway Bridge,Port of Entry,Harlingen Immigration Court,"2009 West Jefferson Ave., Suite 300, Harlingen, TX 78550",POINT(-97.718684038948 26.194727004596)
Hidalgo International Bridge,Port of Entry,Harlingen Immigration Court,"2009 West Jefferson Ave., Suite 300, Harlingen, TX 78550",POINT(-97.718684038948 26.194727004596)


In [39]:
%sql select * from aor_to_counties limit 10

 * postgresql://localhost:5432/tx_immigration_enforcement
10 rows affected.


county,aor_nam,offc_nm,geom,aland,awater
Crockett,Dallas,Dallas,...,7270935061.0000000000000,53329.000000000000000
Scurry,Dallas,Dallas,0106000020AD1000000100000001030000000100000029000000E97FB9162D4B59C0B7B75B920350404044334FAE294B59C0C93EC8B2605040409D47C5FF1D4B59C098874CF910704040AFCC5B751D4B59C046EBA86A827A404023F609A0184B59C009C38025577B4040698B6B7C264659C08D7C5EF1D47B404074EACA67794259C07C613255307C40408658FD11864159C09204E10A287C4040F27A30293E4059C02849D74CBE7B4040B80721205F3F59C0A471A8DF857B4040B97020240B3959C09BC761307F7B4040A7CCCD37A23859C087A3AB74777B40405C1B2AC6F92959C0DCDAC2F3527B40406B44300E2E2A59C0349F73B7EB5D4040CB4C69FD2D2A59C08066101FD85D4040132D793C2D2A59C0AF963B33C15A40406B2A8BC22E2A59C06BF294D574574040DD257156442A59C038A0A52BD84E4040475A2A6F472A59C0F9F6AE415F44404017D68D77472A59C06D00362042444040C93846B2472A59C02219726C3D43404020D099B4A92C59C0F3C81F0C3C4340406536C824233259C0BBB7223141434040996379573D3359C0B91B446B45434040A4E36A64573459C0D49810734943404082531F48DE3959C086014BAE6243404020D26F5F073B59C04C88B9A46A4340401250E108523B59C0F2086EA46C4340406919A9F7543B59C0F2086EA46C43404086C8E9EBF93B59C0A2CF471971434040C47AA356983C59C0A0336953754340402B1895D4093E59C06C43C5387F434040FFCEF6E80D3F59C0683F52448643404027A3CA30EE3F59C0FAB836548C4340403E05C078064059C0EB6E9EEA904340405CC64D0D344259C044520B25934340400F7BA180ED4A59C0EE0A7DB08C434040F54718062C4B59C024809BC58B4340408065A549294B59C0CC28965B5A4F40402B89EC832C4B59C0593673486A4F4040E97FB9162D4B59C0B7B75B9203504040,2345090475.0000000000000,5436957.000000000000000
Briscoe,Dallas,Dallas,0106000020AD1000000100000001030000000100000025000000BB809719365E59C0C2DB8310903741402A3BFDA02E5E59C0FEEF880AD54341407099D365315E59C03E247CEF6F444140F4C5DE8B2F5E59C06CCEC133A14F41405B41D3122B5E59C0A3B08BA207504140DE054A0A2C5E59C001DE02098A534140F54718062C5E59C00A4B3CA06C564140540262122E5E59C05A80B6D5AC5F414093399677D55B59C02B306475AB5F4140635E471CB25859C00F0A4AD1CA5F4140F9872D3D9A5159C0B85A272EC75F4140399D64ABCB4C59C0508BC1C3B45F414043C9E4D4CE4559C071AE6186C65F4140C26D6DE1793C59C0ED9BFBABC75F4140CC99ED0A7D3C59C05532005471514140EF6FD05E7D3C59C0B6D617096D5141409624CFF57D3C59C077BAF3C4735041401A51DA1B7C3C59C0560E2DB29D4F414047054EB6813C59C04419AA622A3F41407BF83251843C59C07E1EA33CF33E4140DE3B6A4C883C59C002D53F88643E414017670C73823C59C04B0169FF033E4140BE6C3B6D8D3C59C0CC43A67C0828414073F6CE68AB3D59C0BAD8B45208284140F31E679AB03D59C03271AB2006284140AAD381ACA74259C06CEA3C2AFE274140BCB20B06D74959C021E9D32AFA274140A515DF50F84A59C027DA5548F9274140959A3DD00A4B59C0F3CCCB61F7274140DBDD03745F4C59C00AF5F411F827414080F0A1444B4E59C0040473F4F8274140D9CF6229925059C052D50451F72741408670CCB2275159C04D69FD2D01284140E3C798BB965159C06440F67AF7274140A75CE15D2E5E59C0156F641EF9274140C9CA2F83315E59C072A8DF85AD2F4140BB809719365E59C0C2DB831090374140,2330991073.0000000000000,4068657.000000000000000
Reagan,Dallas,Dallas,0106000020AD100000010000000103000000010000001300000069FE98D6A67159C00BB6114F761F3F40FA264D83A27159C0ADF6B0170A203F40253CA1D79F7159C023A12DE7525C3F40BEC0AC50A47159C09D0DF967065D3F40F88A6EBDA67159C070438CD7BCA63F40B282DF86185159C08578245E9EA63F40D1E97937165159C0475854C4E9A43F406C5B94D9205159C07D772B4B748E3F40ED832C0B265159C0A3C9C51858873F40EE224C512E5159C0CF49EF1B5F7B3F4094895B05315159C0E5637781927A3F40FAB660A92E5159C0C3D5011077793F40520B2593535159C08B51D7DAFB583F40310BED9C665159C03CFA5FAE454B3F404E7E8B4E965159C03BFDA02E52143F406E895C70065859C0C685032159143F40EB1ED95C355D59C0E481C8224D143F40C51C041DAD7159C07BBE66B96C143F4069FE98D6A67159C00BB6114F761F3F40,3044049119.0000000000000,1792716.000000000000000
Bowie,Dallas,Dallas,...,2291982022.0000000000000,98385599.000000000000000
Hockley,Dallas,Dallas,0106000020AD1000000100000001030000000100000030000000F75AD07B63A759C0151F9F909DE94040ABB2EF8AE0A059C0236937FA98E940409A417C60C7A059C0E179A9D898E9404067614F3BFC9B59C051853FC39BE94040381268B0A99959C08109DCBA9BE94040F19BC24A059859C0813D26529AE94040E3E0D231E79759C0350873BB97E94040D7A3703D0A9259C059DE550F98E94040F04DD36

## Import Area of Responsibility Shapefiles from Deportation Data Project at UC Berkeley

In [18]:
aor = gpd.read_file("/home/jovyan/work/GGIS 407/Final Project/data/ice-aor-county-shp.shp")

In [19]:
aor.head()

,STATEFP,COUNTYF,COUNTYN,GEOIDFQ,GEOID,NAME,NAMELSA,STUSPS,STATE_N,LSAD,ALAND,AWATER,offc_nm,notes,aor_nam,geometry
0,06,079,00277304,0500000US06079,06079,San Luis Obispo,San Luis Obispo County,CA,California,06,8.549141e+09,815650229.0,Los Angeles,None,Los Angeles,"POLYGON ((-121.34636 35.79518, -121.24498 35.7..."
1,06,081,00277305,0500000US06081,06081,San Mateo,San Mateo County,CA,California,06,1.161921e+09,757163219.0,San Francisco,Counties in Northern California not in the LA ...,San Francisco,"POLYGON ((-122.52085 37.59418, -122.51533 37.5..."
2,06,091,00277310,0500000US06091,06091,Sierra,Sierra County,CA,California,06,2.468695e+09,23299110.0,San Francisco,Counties in Northern California not in the LA ...,San Francisco,"POLYGON ((-121.05748 39.53999, -121.05643 39.5..."
3,06,067,00277298,0500000US06067,06067,Sacramento,Sacramento County,CA,California,06,2.500063e+09,75323439.0,San Francisco,Counties in Northern California not in the LA ...,San Francisco,"POLYGON ((-121.86254 38.06795, -121.86160 38.0..."
4,06,107,00277318,0500000US06107,06107,Tulare,Tulare County,CA,California,06,1.249384e+10,37260863.0,San Francisco,Counties in Northern California not in the LA ...,San Francisco,"POLYGON ((-119.56647 36.49434, -119.56366 36.4..."


In [20]:
print(aor.columns.tolist())
print(aor.crs)

['STATEFP', 'COUNTYF', 'COUNTYN', 'GEOIDFQ', 'GEOID', 'NAME', 'NAMELSA', 'STUSPS', 'STATE_N', 'LSAD', 'ALAND', 'AWATER', 'offc_nm', 'notes', 'aor_nam', 'geometry']
epsg:4269


In [21]:
texas_aor = aor[aor['STATE_N'] == 'Texas'] 

In [26]:
print(texas_aor['aor_nam'].unique())
print("Counties Per Each Area of Responsibility (AOR): ")
print(texas_aor.groupby('aor_nam')['NAMELSA'].count()) 

['Houston' 'Dallas' 'Harlingen' 'El Paso' 'San Antonio']
Counties Per Each Area of Responsibility (AOR): 
aor_nam
Dallas         128
El Paso         18
Harlingen       15
Houston         57
San Antonio     36
Name: NAMELSA, dtype: int64


## Find and retrieve nearest nodes for immigration courts, facilities, and towers on OSMnx network graphs:

In [23]:
#courts_gdf
courts = gpd.read_file("/home/jovyan/work/GGIS 407/Final Project/data/processed/courts_gdf.geojson")
detention_centers = gpd.read_file("/home/jovyan/work/GGIS 407/Final Project/data/processed/EOIR_complete_gdf.geojson")
counties_287g_enforcement_types = gpd.read_file("/home/jovyan/work/GGIS 407/Final Project/data/processed/tx_287g_counties_gdf.geojson")                

In [24]:
facilities = pd.read_csv("/home/jovyan/work/GGIS 407/Final Project/data/facilities.csv")
courts = pd.read_csv("/home/jovyan/work/GGIS 407/Final Project/data/texas_immigration_judges.csv")

print(facilities.columns.tolist())
print(courts.columns.tolist())

['detention_facility_code', 'detention_facility_name', 'address', 'city', 'county', 'state', 'zip', 'aor', 'latitude', 'longitude', 'type_detailed', 'type_grouped']
['Court', 'Judge_Name', 'Initials', 'WebEx_Link', 'Access_Code']


## Now get road networks for each AOR within buffer zone of points of interest nodes

In [ ]:
# G_drive_harlingen = texas_aor[texas_aor['aor_nam'] == 'Harlingen']
# harlingen_union_poly = G_drive_harlingen.geometry.unary_union
# G_drive_harlingen = ox.graph_from_polygon(harlingen_union_poly, network_type='drive')
# G_drive_harlingen = ox.add_edge_speeds(G_drive_harlingen)
# G_drive_harlingen = ox.add_edge_travel_times(G_drive_harlingen)
# ox.save_graphml(G_drive_harlingen, filepath="data/G_drive_harlingen.graphml")
# print("Done!")

amenity_values = ox.geometries_from_bbox(north, south, east, west, tags = {"amenity": True})

# Extract the "amenity" column, remove missing values, and list all unique amenity types found in aor
amenity_values['amenity'].dropna().unique()

In [41]:
G_drive_eptx = texas_aor[texas_aor['aor_nam'] == 'El Paso']
el_paso_union_poly = G_drive_elpaso.geometry.unary_union

G_drive_satx = texas_aor[texas_aor['aor_nam'] == 'San Antonio']
san_antonio_union_poly = G_drive_satx.geometry.unary_union

G_drive_houston = texas_aor[texas_aor['aor_nam'] == 'Houston']
houston_union_poly = G_drive_houston.geometry.unary_union

G_drive_dallas = texas_aor[texas_aor['aor_nam'] == 'Dallas']
dallas_union_poly = G_drive_dalllas.geometry.unary_union

In [42]:
G_drive_elpaso = ox.graph_from_polygon(el_paso_union_poly, network_type='drive')

In [ ]:
G_drive_elpaso = ox.add_edge_speeds(G_drive_elpaso)
G_drive_elpaso = ox.add_edge_travel_times(G_drive_elpaso)
ox.save_graphml(G_drive_elpaso, filepath="data/G_drive_elpaso.graphml")
print("Done!")

In [ ]:
nearest = min(poi_points, key=lambda p: p.distance(nodes_drive.loc[origin_node].geometry))
distance_m = nearest.distance(nodes_drive.loc[origin_node].geometry)
nearest_rest_name = restaurants.loc[restaurants.geometry == nearest, 'name'].values[0]

route_distance = ox.shortest_path(G_drive_harlingen, origin_node, dest_node, weight='length')
route_time = ox.shortest_path(G_drive_harlingen, origin_node, dest_node, weight='travel_time')


millenium_geom = nodes_proj.loc[origin_node].geometry
rest_geom = nodes_proj.loc[dest_node].geometry

In [ ]:
# Project to Web Mercator (required for basemap)
G_drive_harlingen_proj = ox.project_graph(G_drive_harlingen, to_crs = "EPSG:3857")



fig, ax = ox.plot_graph(G_drive_harlingen_proj, bgcolor='white', node_size=0, 
                        edge_linewidth=0.5,edge_color='darkblue',
                        show=False, close=False)



# Add basemap
# cx.add_basemap(ax, crs="EPSG:3857")

# Title and layout
ax.set_title('Harlingen Driving Network')
plt.tight_layout()
plt.show()


In [ ]:
for aor_name, group in texas_aor.groupby('aor_nam'):
    filepath = f"data/G_drive_{aor_name.lower()}.graphml"
    
    if os.path.exists(filepath):
        print(f"Skipping {aor_name} — already exists")
        continue
    
    print(f"Downloading {aor_name}...")
    union_poly = group.geometry.unary_union
    G_drive = ox.graph_from_polygon(union_poly, network_type='drive')
    G_drive = ox.add_edge_speeds(G_drive)
    G_drive = ox.add_edge_travel_times(G_drive)
    ox.save_graphml(G_drive, filepath=filepath)
    print(f"Saved {aor_name}!")